## Chapter 4: "Data Structures and Algorithms"

This week's exercise has a total of **9 tasks**, worth **11 points**.

In case you want to, you can turn on Autosave in Colab, so your progress is saved. You can execute this, and any other cell by pressing Shift+Enter, or by pressing the little play button next to it.

In [ ]:
%autosave 60

### Chapter 4.0: "Choosing the Right Data Structure"

Before we dive into graphs, let's warm up with a practical question: *you receive some data - how do you store it?*

The lecture gave us a simple cheatsheet:
- **Order matters, duplicates allowed, change often?** -> List
- **Fixed record of mixed values, or a dictionary key?** -> Tuple
- **Only "is it in there?" and uniqueness?** -> Set
- **Look something up by a name or id?** -> Dictionary
- **Numbers in bulk, maths on whole tables?** -> Array / Matrix / Tensor
- **Objects interconnected?** -> Graph (next section!)

Let's put this into practice.

In [3]:
# A small clinic sends us a text file with today's data. Here it is:

raw_data = """
PATIENTS: Alice, Bob, Charlie, Alice
DOCTOR_SCHEDULE: Dr. Smith, 09:00, Room 3
ROOMS: Alice->101, Bob->102, Charlie->103
TEMPERATURES:
36.5 37.1 36.8
38.2 36.9 37.4
"""

print(raw_data)


PATIENTS: Alice, Bob, Charlie, Alice
DOCTOR_SCHEDULE: Dr. Smith, 09:00, Room 3
ROOMS: Alice->101, Bob->102, Charlie->103
TEMPERATURES:
36.5 37.1 36.8
38.2 36.9 37.4



Let's look at each line and think about what we need to do with it:

| Line | What we need | Best structure |
|------|-------------|----------------|
| `PATIENTS` | Keep order, Alice appears twice, we'll add more later | ? |
| `DOCTOR_SCHEDULE` | Fixed record: (name, time, room) - won't change today | ? |
| `ROOMS` | Look up "which room is Alice in?" by name | ? |
| `TEMPERATURES` | 2 patients x 3 measurements, we'll compute averages | ? |

Think about it before scrolling down!

**Task 1 (1 point)**:
Parse the `raw_data` string above and store each section in the most appropriate Python data structure:

1. `patients` - a **list** of strings (order matters, duplicates allowed)
2. `schedule` - a **tuple** of 3 elements: `(doctor_name, time, room)`
3. `rooms` - a **dictionary** mapping patient name -> room number (int)
4. `temps` - a **NumPy array** of shape `(2, 3)` (2 patients, 3 measurements)

After creating them, print each one and add a one-line comment explaining *why* you chose that structure (e.g., "list because we need to preserve order and allow duplicates").

In [4]:
# TASK 1 SOLUTION
import numpy as np

PATIENTS = ["Alice", "Bob", "Charlie", "Alice"]

SCHEDULE = ("Dr. Smith", "09:00", "Room 3")

ROOMS = {"Alice": 101, "Bob": 102, "Charlie": 103}

TEMPS = np.array([[36.5, 37.1, 36.8], [38.2, 36.9, 37.4]])

print(PATIENTS)
print(SCHEDULE)
print(ROOMS)
print(TEMPS)

['Alice', 'Bob', 'Charlie', 'Alice']
('Dr. Smith', '09:00', 'Room 3')
{'Alice': 101, 'Bob': 102, 'Charlie': 103}
[[36.5 37.1 36.8]
 [38.2 36.9 37.4]]


### Chapter 4.1: "Graph Representations"

A **graph** $G = (V, E)$ is a set of *vertices* (nodes) connected by *edges*. In Python, we most often store a graph as a **dictionary of lists** (adjacency list) or a **dictionary of dictionaries** (weighted adjacency list).

Why does this matter? Because the *same* problem can take 1 second or 1 day depending on which structure you pick. For sparse graphs (few edges), an adjacency list is compact and fast. For dense graphs, a matrix is easier to index into.

Let's set up the graph we will use throughout this tutorial:

In [18]:
# This is our unweighted graph, stored as an adjacency list (dict of lists).
# g["A"] tells us: "A is connected to B and C."

g = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B", "D"],
    "D": ["B", "C"]
}

# Let's check a few things:
print(g["B"])            # neighbors of B -> ["A", "C", "D"]
print("C" in g["A"])     # is there an edge A-C? -> True
print(len(g["B"]))       # degree of B -> 3

['A', 'C', 'D']
True
3


In [6]:
# For a *weighted* graph, we store the weight next to each neighbor:
# wg["A"]["C"] means "the edge from A to C has weight 2."

wg = {
    "A": {"B": 4, "C": 2},
    "B": {"A": 4, "C": 1, "D": 5},
    "C": {"A": 2, "B": 1, "D": 8},
    "D": {"B": 5, "C": 8}
}

print(wg["A"]["C"])      # weight of edge A-C -> 2
for v, w in wg["B"].items():
    print(f"  B -> {v} (weight {w})")

2
  B -> A (weight 4)
  B -> C (weight 1)
  B -> D (weight 5)


In [7]:
# Another common representation is the *edge list*: just a list of (weight, u, v) tuples.
# This is what you get when you read a graph from a file.

edges = [(4, "A", "B"), (2, "A", "C"), (1, "B", "C"),
         (5, "B", "D"), (8, "C", "D")]

# We can convert an edge list into a weighted adjacency list:
wg_from_edges = {}
for w, u, v in edges:
    wg_from_edges.setdefault(u, {})[v] = w
    wg_from_edges.setdefault(v, {})[u] = w  # undirected: add both directions

print(wg_from_edges)
# Should be identical to wg above.

{'A': {'B': 4, 'C': 2}, 'B': {'A': 4, 'C': 1, 'D': 5}, 'C': {'A': 2, 'B': 1, 'D': 8}, 'D': {'B': 5, 'C': 8}}


**Task 2 (1 point)**:
Write a function `edge_list_to_adjacency(edges)` that takes a list of `(u, v)` tuples (unweighted, undirected) and returns an adjacency list (a dictionary of lists). Make sure that if edge `("A", "B")` is in the list, then both `g["A"]` contains `"B"` *and* `g["B"]` contains `"A"`. Do not add duplicate neighbors.

In [8]:
# TASK 2 SOLUTION

def edge_list_to_adjacency(edges):
    g = {}
    for u, v in edges:
        g.setdefault(u, [])
        g.setdefault(v, [])
        if v not in g[u]:
            g[u].append(v)
        if u not in g[v]:
            g[v].append(u)
    return g


# Test
edges = [("A", "B"), ("A", "C"), ("B", "C"), ("B", "D"), ("A", "B")]  # ("A","B") doppelt
g = edge_list_to_adjacency(edges)
print(g)


{'A': ['B', 'C'], 'B': ['A', 'C', 'D'], 'C': ['A', 'B'], 'D': ['B']}


### Chapter 4.2: "Graph Traversal: BFS and DFS"

The most fundamental question about a graph is: *which vertices can I reach from here, and in what order?*

The answer is a **traversal**. You keep two things:
- A collection of vertices **still to visit** (a queue for BFS, a stack for DFS)
- A set of vertices **already seen** (so you don't visit them twice)

The *only* difference between BFS and DFS is **which waiting vertex you pick next**:
- **BFS** (Breadth-First Search): oldest first -> a *queue* -> explores layer by layer
- **DFS** (Depth-First Search): newest first -> a *stack* -> goes as deep as possible, then backtracks

Both run in $O(n + m)$ with an adjacency list, where $n$ is the number of vertices and $m$ the number of edges.

In [9]:
# Let's look at BFS first. We use collections.deque as our queue,
# because deque.popleft() is O(1), while list.pop(0) is O(n).

from collections import deque

def bfs(g, start):
    """Return a dict mapping each reachable vertex to its distance (in edges) from start."""
    dist = {start: 0}       # also serves as the "seen" set
    queue = deque([start])
    while queue:
        u = queue.popleft()  # oldest first
        for v in g[u]:
            if v not in dist:
                dist[v] = dist[u] + 1
                queue.append(v)
    return dist

print(bfs(g, "A"))
# Expected: {'A': 0, 'B': 1, 'C': 1, 'D': 2}

{'A': 0, 'B': 1, 'C': 1, 'D': 2}


In [10]:
# DFS uses a plain list as a stack: append to push, pop() to take the newest.

def dfs_iter(g, start):
    """Return the order in which vertices are visited by an iterative DFS."""
    seen = set()
    stack = [start]
    order = []
    while stack:
        u = stack.pop()       # newest first
        if u in seen:
            continue
        seen.add(u)
        order.append(u)
        stack.extend(g[u])    # push all neighbors
    return order

print(dfs_iter(g, "A"))
# One possible output: ['A', 'C', 'D', 'B']
# (The exact order depends on the order of neighbors in the list.)

['A', 'C', 'B', 'D']


**Task 3 (1 point)**:
Write a function `bfs_order(g, start)` that returns the *order* in which vertices are visited by BFS (a list, not a distance dict). Use the same `g` graph from above. Start at `"A"`.

In [11]:
# TASK 3 SOLUTION

from collections import deque

def bfs_order(g, start):
    seen = {start}
    queue = deque([start])
    order = []
    while queue:
        u = queue.popleft()
        order.append(u)
        for v in g[u]:
            if v not in seen:
                seen.add(v)
                queue.append(v)
    return order

print(bfs_order(g, "A"))
# ['A', 'B', 'C', 'D']

['A', 'B', 'C', 'D']


**Task 4 (1 point)**:
Write a *recursive* version of DFS: `dfs_recursive(g, node, seen)`. It should add `node` to `seen`, then recursively visit every neighbor not yet in `seen`. Return `seen`. Test it on our graph starting at `"A"`.

In [12]:
# TASK 4 SOLUTION

def dfs_recursive(g, node, seen):
    seen.add(node)
    for v in g[node]:
        if v not in seen:
            dfs_recursive(g, v, seen)
    return seen

print(dfs_recursive(g, "A", set()))
# {'A', 'B', 'C', 'D'}   (Reihenfolge der Ausgabe kann variieren, da Set)

{'D', 'C', 'A', 'B'}


### Chapter 4.3: "Connected Components"

A **connected component** is a maximal group of vertices where every vertex can reach every other. In a disconnected graph, there are multiple components.

The trick is simple: run a traversal from every vertex you haven't seen yet. Each traversal gives you one component.

In [13]:
def components(g):
    """Return a list of sets, one per connected component."""
    seen = set()
    comps = []
    for v in g:
        if v not in seen:
            comp = set(dfs_iter(g, v))
            seen |= comp
            comps.append(comp)
    return comps

# Our main graph is fully connected, so we get one component:
print(components(g))
# [{'A', 'B', 'C', 'D'}]

# Let's try a disconnected graph:
g_two = {"A": ["B"], "B": ["A"], "C": [], "D": ["E"], "E": ["D"]}
print(components(g_two))
# [{'A', 'B'}, {'C'}, {'D', 'E'}]

[{'D', 'A', 'B', 'C'}]
[{'A', 'B'}, {'C'}, {'D', 'E'}]


**Task 5 (1 point)**:
Write a function `num_components(g)` that returns the *number* of connected components in a graph. You can reuse `components()` or write your own logic. Test it on `g` (should be 1) and `g_two` (should be 3).

In [14]:
# TASK 5 SOLUTION

def num_components(g):
    return len(components(g))

print(num_components(g))      # 1
print(num_components(g_two))  # 3



def num_components(g):
    seen = set()
    count = 0
    for v in g:
        if v not in seen:
            seen |= set(dfs_iter(g, v))
            count += 1
    return count

print(num_components(g))      # 1
print(num_components(g_two))  # 3

1
3
1
3


### Chapter 4.4: "Recursion"

A **recursive** function solves a problem by calling itself on a *smaller* input. Two things are mandatory:
1. **Base case**: the smallest input, answered directly. This stops the recursion.
2. **Recursive case**: reduce the problem, call yourself, combine the results.

Classic example: factorial. $n! = n \times (n-1)!$, with $0! = 1$.

In [15]:
def factorial(n):
    if n == 0:                    # base case
        return 1
    return n * factorial(n - 1)   # recursive case

print(factorial(4))
# = 4 * factorial(3)
# = 4 * 3 * factorial(2)
# = 4 * 3 * 2 * factorial(1)
# = 4 * 3 * 2 * 1 * factorial(0)
# = 24

24


In [2]:
# Recursion also works on lists. The sum of a list is:
# first element + sum of the rest.
# Base case: empty list has sum 0.

def total(xs):
    if not xs:                    # empty list: base case
        return 0
    return xs[0] + total(xs[1:])  # first element + sum of the rest

print(total([1, 2, 3, 4]))  # 10

10


**Task 6 (1 point)**:
Write a recursive function `count_edges(g, node, seen)` that counts how many unique undirected edges are in the connected component containing `node`. Each edge (e.g., A-B) must be counted exactly once, regardless of which endpoint visits it first. Test it on `g` starting at `"A"` (should be 5: A-B, A-C, B-C, B-D, C-D).

In [19]:
# TASK 6 SOLUTION

g = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B", "D"],
    "D": ["B", "C"]
}

def count_edges(g, node, seen):
    # Kanten zu Nachbarn zählen, die VOR diesem Knoten besucht wurden
    count = 0
    for v in g[node]:
        if v in seen:
            count += 1
    seen.add(node)

    # rekursiv weiter in die Tiefe
    for v in g[node]:
        if v not in seen:
            count += count_edges(g, v, seen)
    return count

print(count_edges(g, "A", set()))
print(count_edges(g_two, "A", set()))  # 1  (nur A-B)

{'A': ['B', 'C'], 'B': ['A', 'C', 'D'], 'C': ['A', 'B', 'D'], 'D': ['B', 'C']}
5
1


### Chapter 4.5: "Memoization and Dynamic Programming"

The Fibonacci sequence looks naturally recursive:
$$F(n) = F(n-1) + F(n-2), \quad F(0) = 0,\; F(1) = 1$$

But naive recursion is *exponentially* slow: `fib(40)` makes about 330 million calls, because the same sub-problems are recomputed over and over.

**Memoization** fixes this: store each result in a dictionary the first time you compute it, then look it up afterwards. This turns $O(2^n)$ into $O(n)$.

**Dynamic programming (DP)** generalizes this idea: solve a problem by combining solutions of *overlapping sub-problems*, each solved once. Two styles:
- **Top-down**: recursion + memoization (what we just saw)
- **Bottom-up**: fill a table with a loop, starting from the smallest sub-problems

In [20]:
# Naive recursive Fibonacci: O(2^n)
def fib_naive(n):
    if n < 2:
        return n
    return fib_naive(n - 1) + fib_naive(n - 2)

# Memoized Fibonacci: O(n)
memo = {}
def fib_memo(n):
    if n < 2:
        return n
    if n not in memo:
        memo[n] = fib_memo(n - 1) + fib_memo(n - 2)
    return memo[n]

print(fib_memo(10))   # 55
print(fib_memo(40))   # 102334155, instant
# fib_naive(40) would take tens of seconds - don't run it!

55
102334155


In [21]:
# Bottom-up DP: no recursion, no stack limit, O(n) time, O(1) memory
def fib_bottomup(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

print(fib_bottomup(10))  # 55
print(fib_bottomup(40))  # 102334155

55
102334155


**Task 7 (1 point)**:
Write a memoized recursive function `fib(n)` (use a dictionary or `functools.lru_cache`). Then write a bottom-up version `fib_iter(n)`. Both must return $F(n)$. Test them on $n = 0, 1, 10, 50$ and make sure they agree.

In [22]:
# TASK 7 SOLUTION

from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)


def fib_iter(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a


for n in [0, 1, 10, 50]:
    print(n, fib(n), fib_iter(n), fib(n) == fib_iter(n))
    assert fib(n) == fib_iter(n)

0 0 0 True
1 1 1 True
10 55 55 True
50 12586269025 12586269025 True


Now let's do a real DP problem where **greedy fails**.

**Fewest coins**: Given coins `[1, 5, 6, 9]` and amount `11`, find the minimum number of coins.

Greedy would pick 9, then 1, then 1 -> **3 coins**. But $5 + 6 = 11$ -> only **2 coins**!

DP recipe:
1. Define the sub-problem: `best[a]` = fewest coins that sum to `a`
2. Recurrence: `best[a] = 1 + min(best[a - c] for every coin c <= a)`
3. Base case: `best[0] = 0`
4. Fill the table from `a = 1` up to `a = amount`
5. Read the answer from `best[amount]`

In [23]:
def min_coins(coins, amount):
    """Return the fewest number of coins needed to make `amount`, or -1 if impossible."""
    best = [0] + [float("inf")] * amount
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                best[a] = min(best[a], best[a - c] + 1)
    return best[amount] if best[amount] != float("inf") else -1

print(min_coins([1, 5, 6, 9], 11))  # 2  (5 + 6)
print(min_coins([1, 5, 6, 9], 3))   # 3  (1 + 1 + 1)
print(min_coins([2, 5], 3))         # -1 (impossible)

2
3
-1


**Task 8 (2 points)**:
Extend `min_coins` to also return the *actual coins used*, not just the count. Write a function `min_coins_with_path(coins, amount)` that returns a tuple `(count, coin_list)`. For example, `min_coins_with_path([1, 5, 6, 9], 11)` should return `(2, [5, 6])` (order doesn't matter). If the amount is impossible, return `(-1, [])`.

In [24]:
# TASK 8 SOLUTION

def min_coins_with_path(coins, amount):
    best = [0] + [float("inf")] * amount
    last = [None] * (amount + 1)          # welche Münze zuletzt gelegt wurde

    for a in range(1, amount + 1):
        for c in coins:
            if c <= a and best[a - c] + 1 < best[a]:
                best[a] = best[a - c] + 1
                last[a] = c               # merken: für Betrag a war c die letzte Münze

    if best[amount] == float("inf"):
        return (-1, [])

    # Münzen zurückverfolgen
    coin_list = []
    a = amount
    while a > 0:
        coin_list.append(last[a])
        a = a - last[a]

    return (best[amount], coin_list)


print(min_coins_with_path([1, 5, 6, 9], 11))  # (2, [5, 6])
print(min_coins_with_path([1, 5, 6, 9], 3))   # (3, [1, 1, 1])
print(min_coins_with_path([2, 5], 3))         # (-1, [])

(2, [5, 6])
(3, [1, 1, 1])
(-1, [])


### Chapter 4.6: "Diagnosing Errors"

Even experienced programmers write bugs. The key is to have a **checklist**:
1. Test on tiny inputs you can solve by hand.
2. Print or inspect intermediate state (queue, stack, dict, table).
3. Check data shapes and types (list vs. set, dict of lists vs. dict of dicts).
4. Check edge cases: empty input, one vertex, disconnected graph.
5. Compare with a slow but obviously correct version.

Common bugs in graph code:
- `KeyError`: vertex missing from the dictionary
- Infinite loop in BFS/DFS: forgot the `seen` set, or marked `seen` too late
- Wrong order: using `list.pop(0)` as a queue (O(n)) instead of `deque.popleft()` (O(1))
- Off-by-one in DP table size (should be `amount + 1`, not `amount`)

In [25]:
# Here are two broken functions. Each has exactly ONE bug.
# Read the code, figure out what's wrong, and fix it.

# BUGGY FUNCTION 1: BFS that never terminates
def bfs_buggy(g, start):
    dist = {start: 0}
    queue = deque([start])
    while queue:
        u = queue.popleft()
        for v in g[u]:
            # BUG: we add v to the queue even if we already visited it,
            # and we never mark it as seen before adding.
            # What should the condition be?
            dist[v] = dist[u] + 1
            queue.append(v)
    return dist

# Don't run this! It will loop forever.
# print(bfs_buggy(g, "A"))  # <- would hang

In [26]:
# BUGGY FUNCTION 2: DP table with off-by-one error
def min_coins_buggy(coins, amount):
    # BUG: table has `amount` entries (indices 0..amount-1),
    # but we need indices 0..amount, so it should be `amount + 1` entries.
    best = [0] + [float("inf")] * (amount - 1)  # too short!
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                best[a] = min(best[a], best[a - c] + 1)
    return best[amount]

# This will raise an IndexError when amount >= 2.
# print(min_coins_buggy([1, 5, 6, 9], 11))  # <- would crash

**Task 9 (2 points)**:
Fix both buggy functions above. Write the corrected versions as `bfs_fixed(g, start)` and `min_coins_fixed(coins, amount)`. Add a short comment explaining what the bug was. Test them on our graph and on `min_coins_fixed([1, 5, 6, 9], 11)` (should return 2).

In [27]:
# TASK 9 SOLUTION

from collections import deque

g = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B", "D"],
    "D": ["B", "C"],
}


# KORRIGIERTE FUNKTION 1
# Fehler: Jeder Nachbar wurde in die Queue gestellt, auch wenn er schon
# besucht war. Dadurch kamen Knoten immer wieder hinein, die Queue wurde
# nie leer, und die Abstände wurden mit falschen Werten überschrieben.
# Korrektur: v nur bearbeiten, wenn es noch nicht in dist steht.
def bfs_fixed(g, start):
    dist = {start: 0}
    queue = deque([start])
    while queue:
        u = queue.popleft()
        for v in g[u]:
            if v not in dist:          # die fehlende Abfrage
                dist[v] = dist[u] + 1
                queue.append(v)
    return dist


# KORRIGIERTE FUNKTION 2
# Fehler: Die Tabelle hatte nur `amount` Einträge (Index 0 bis amount-1),
# deshalb führte best[amount] zu einem IndexError.
# Korrektur: `amount + 1` Einträge anlegen, damit Index 0 bis amount existiert.
def min_coins_fixed(coins, amount):
    best = [0] + [float("inf")] * amount    # vorher: * (amount - 1)
    for a in range(1, amount + 1):
        for c in coins:
            if c <= a:
                best[a] = min(best[a], best[a - c] + 1)
    return best[amount] if best[amount] != float("inf") else -1


# Tests
print(bfs_fixed(g, "A"))                 # {'A': 0, 'B': 1, 'C': 1, 'D': 2}
print(min_coins_fixed([1, 5, 6, 9], 11)) # 2

assert bfs_fixed(g, "A") == {"A": 0, "B": 1, "C": 1, "D": 2}
assert min_coins_fixed([1, 5, 6, 9], 11) == 2
assert min_coins_fixed([2, 5], 3) == -1
print("Alle Tests bestanden.")

{'A': 0, 'B': 1, 'C': 1, 'D': 2}
2
Alle Tests bestanden.


**Don't forget to commit your solutions to GitHub!**